In [1]:
import pandas as pd
from sklearn import preprocessing, pipeline, ensemble, compose
import datasets
import os

paths = {
    'real': '/hdd3/sonia/data/adult.csv',
    'dgpt2': '/hdd3/sonia/be_great/ckpts/dgpt2/adult-allcol/samples.csv',
    'moe': '/hdd3/sonia/be_great/ckpts/moe/dgpt2/adult-allcol/jul21/samplesclean.csv',
    'greatdgpt2': '/hdd3/sonia/be_great/ckpts/dgpt2-greatclean.csv',
    'moegreatdgpt2': '/hdd3/sonia/be_great/ckpts/great/adult/moegreatdgpt2-aug12.csv',
    'mhmoegreatdgpt2-0': '/hdd3/sonia/be_great/ckpts/moemh/dgpt2/adult-allcol/aug14-0.csv',
    'mhmoegreatdgpt2-1': '/hdd3/sonia/be_great/ckpts/moemh/dgpt2/adult-allcol/aug14-1.csv',
    'mhmoegreatdgpt2-2': '/hdd3/sonia/be_great/ckpts/moemh/dgpt2/adult-allcol/aug14-2.csv',
    'fairgan': '/hdd3/sonia/be_great/ckpts/tabfairgan/adult-april.csv',
}
rs = 4 # random state
train_frac = 0.75

ords = ['workclass', 'education', 'marital-status', 'occupation', 
        'relationship', 'race', 'sex', 'native-country'] # MUST BE IN ORDER
nums = ['age', 'fnlwgt', 'education-num', 'capital-gain', 'capital-loss', 'hours-per-week', ]
labs = ['income']

/hdd2/sonia/miniconda3/envs/great/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
datadict = {k:pd.read_csv(v) for (k,v) in paths.items()}
print([k+': '+str(df.shape) for (k, df) in datadict.items()]) # print shape of each dataset

min_dataset_size = min([len(df) for df in datadict.values()])
train_size = int(min_dataset_size*train_frac)
categoriesdict = dict() # collect all unique values for each of the ordinal columns
for (k, df) in datadict.items():
    print(k)
    # remove extra spaces around strings, eg ' dog' -> 'dog'
    df = df.map(lambda x: x.strip() if type(x) == str else x)
    for col in ords:
        categoriesdict[col] = categoriesdict.get(col, []) + df[col].unique().tolist()
        
    if k == 'real':
        datadict[k] = {'train':df, 'test': df}
    else:
        # sample to min dataset size, shuffle, ensure cols in order w no extra cols:
        df = df.sample(min_dataset_size, random_state=rs, ignore_index=True)[ords+nums+labs]
        datadict[k] = {'train':df.iloc[:train_size, :],
                    'test': df.iloc[train_size:, :]}
print(f'sampled {min_dataset_size} rows from each dataset, made trainsets of {train_size} rows each')

categories = []
for col in ords:
    categories.append(list(set(categoriesdict[col])))
ordenc = preprocessing.OrdinalEncoder(categories=categories)
numenc = preprocessing.StandardScaler()
lb = preprocessing.LabelBinarizer()

['real: (48842, 15)', 'dgpt2: (5935, 15)', 'moe: (9106, 15)', 'greatdgpt2: (9815, 15)', 'moegreatdgpt2: (9997, 15)', 'mhmoegreatdgpt2-0: (9995, 15)', 'mhmoegreatdgpt2-1: (9996, 15)', 'mhmoegreatdgpt2-2: (9996, 15)', 'fairgan: (32561, 15)']
real
dgpt2
moe
greatdgpt2
moegreatdgpt2
mhmoegreatdgpt2-0
mhmoegreatdgpt2-1
mhmoegreatdgpt2-2
fairgan
sampled 5935 rows from each dataset, made trainsets of 4451 rows each


In [3]:
# make random forest sklearn pipeline
def create_pipeline(trainset):
    rfc = ensemble.RandomForestClassifier(n_estimators=10, max_depth=4, random_state=rs)
    preprocessing_pipeline = compose.ColumnTransformer([
        ("ordinal_preprocessor", ordenc, ords),
        ("numerical_preprocessor", numenc, nums),
    ])
    complete_pipeline = pipeline.Pipeline([
        ("preprocessor", preprocessing_pipeline),
        ("estimator", rfc)
    ])
    
    preprocessed_labels = lb.fit_transform(trainset[labs].values.ravel()).ravel()
    complete_pipeline.fit(trainset[ords+nums], preprocessed_labels)
    return complete_pipeline

rfdict = {}
for src in datadict.keys():
    print(src)
    rfdict[src] = create_pipeline(datadict[src]['train'])

real
dgpt2
moe
greatdgpt2
moegreatdgpt2
mhmoegreatdgpt2-0
mhmoegreatdgpt2-1
mhmoegreatdgpt2-2
fairgan


In [4]:
for data in datadict.keys():
    print(data)
    labels = lb.fit_transform(datadict[data]['test'][labs])
    for model in rfdict.keys():
        score = rfdict[model].score(datadict[data]['test'][ords+nums], labels)
        print(f'{model} on {data}: \t\t\t{score}')
    print('\n')

real
real on real: 			0.838786290487695
dgpt2 on real: 			0.8287948896441587
moe on real: 			0.7953400761639573
greatdgpt2 on real: 			0.8370459850128987
moegreatdgpt2 on real: 			0.8225912124810614
mhmoegreatdgpt2-0 on real: 			0.8278326030875066
mhmoegreatdgpt2-1 on real: 			0.8190287048032431
mhmoegreatdgpt2-2 on real: 			0.8181073666107039
fairgan on real: 			0.824495311412309


dgpt2
real on dgpt2: 			0.7621293800539084
dgpt2 on dgpt2: 			0.7978436657681941
moe on dgpt2: 			0.7668463611859838
greatdgpt2 on dgpt2: 			0.7681940700808625
moegreatdgpt2 on dgpt2: 			0.7405660377358491
mhmoegreatdgpt2-0 on dgpt2: 			0.7506738544474394
mhmoegreatdgpt2-1 on dgpt2: 			0.7216981132075472
mhmoegreatdgpt2-2 on dgpt2: 			0.7142857142857143
fairgan on dgpt2: 			0.7196765498652291


moe
real on moe: 			0.7722371967654986
dgpt2 on moe: 			0.7654986522911051
moe on moe: 			0.7964959568733153
greatdgpt2 on moe: 			0.7681940700808625
moegreatdgpt2 on moe: 			0.77088948787062
mhmoegreatdgpt2-0 on moe